In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.5 MB/s eta 0:00:00


In [ ]:
# load data
data = Planetoid(root='data', name='Cora')[0]

# model
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = GCNConv(data.num_features, 16)
        self.c2 = GCNConv(16, 7)   # 7 classes in Cora

    def forward(self, x, edge_index):
        x = self.c1(x, edge_index)
        x = F.relu(x)
        x = self.c2(x, edge_index)
        return x
model = GCN()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

# training
for i in range(100):
    model.train()
    opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    opt.step()

# testing
model.eval()
out = model(data.x, data.edge_index)
pred = out.argmax(dim=1)

acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
print("Accuracy:", acc)

Processing...
Done!


Accuracy: 0.786


In [ ]:
from torch_geometric.nn import SAGEConv

class SAGE(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = SAGEConv(data.num_features, 16)
        self.c2 = SAGEConv(16, 7)

    def forward(self, x, edge_index):
        x = self.c1(x, edge_index)
        x = F.relu(x)
        x = self.c2(x, edge_index)
        return x

model = SAGE()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for i in range(100):
    model.train()
    opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    opt.step()

model.eval()
out = model(data.x, data.edge_index)
pred = out.argmax(dim=1)

acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
print("Accuracy:", acc)

Accuracy: 0.722
